# Etapas principales en un sistema de aprendizaje de máquina

1. Observar el problema (ayuda interdisciplinar)
2. Obtener los datos
3. *Análisis exploratorio incial (visualizar los datos y estadística descriptiva básica)*
4. *Preparar los datos para los algoritmos de aprendizaje de máquina (evaluación, preproceso, caracterización, aprendizaje)*
5. *Seleccionar un modelo y entrenar*
6. *Sintonizar el modelo escogido*
7. Presentar la solución
8. Lanzar la solución, monitorear y mantener el sistema de aprendizaje de máquina.

# Problema a resolver

**Objetivo**: predecir que zonas de la red se verán afectadas para poder minimizar el Índice Unificado de Tensión e Interrupciones(UITI).

## Obtención de los datos

In [1]:
# !pip install geopandas
# !pip install contextily

# import geopandas as gpd
# import contextily as ctx
# import matplotlib.pyplot as plt

# from sklearn.impute import SimpleImputer
# from sklearn.model_selection import train_test_split
# from sklearn.svm import SVR
# from sklearn.preprocessing import StandardScaler
# from sklearn.metrics import mean_squared_error

import cupy as cp             # Para arrays en GPU, reemplazando a NumPy
import cudf as cu_df          # Para DataFrames en GPU, reemplazando a Pandas
from cuml.model_selection import train_test_split
# from cuml.svm import SVR
from cuml.ensemble import RandomForestRegressor
from cuml.preprocessing import StandardScaler, SimpleImputer
from cuml.metrics import mean_squared_error

In [2]:
path = "/kaggle/input/powergrid-assets-ml-dataset/"

# raw_data = pd.read_csv(path + "dataset_modificado.csv")
# raw_data.head()
# raw_data.iloc[:, 15: 17]
# raw_data.info()

raw_data = cu_df.read_csv(path + "dataset_modificado.csv")

## Análisis exploratorio de los datos

In [3]:
# gdf = gpd.GeoDataFrame(raw_data, geometry=gpd.points_from_xy(raw_data.iloc[:, 15], raw_data.iloc[:, 16]), crs="EPSG:4326")

# # Create a matplotlib figure and axes
# fig, ax = plt.subplots(1, 1, figsize=(10, 10))

# # Plot the gdf GeoDataFrame on the created axes
# gdf.plot(ax=ax, markersize=1, color='red')

# # Add a basemap to the plot
# ctx.add_basemap(ax, crs=gdf.crs)

# # Set the title and axis labels
# ax.set_title('Geographic Distribution of Points with Basemap')
# ax.set_xlabel('Longitude')
# ax.set_ylabel('Latitude')

# # Display the plot
# plt.show()

## Preparación de los datos

Para motivos de un análisis inicial, usaremos solamente los datos numéricos del dataset.

In [4]:
# data = raw_data.copy()
# data = data.drop(data.columns[0], axis = 1)
# data = data.drop(columns = ['CODIGO', 'TIPO', 'FECHA', 'CALIBRE_F', 'MATERIAL_F', 'AISLAMIENTO_F', 'G_N', 'CALIBRE_NEUTRO', 'TIPO_TAXONOMIA', 'CLASE', 'ALTURA', 'CANTIDAD_TIERRA'])

# # Ahora se quita la variable que se quiere predecir
# target = ['UITI']
# y = data[target].values.astype('float32')
# data.drop(target, axis = 1, inplace = True)

# # Se imputan valores faltantes con la media
# imputer = SimpleImputer(strategy = 'mean')
# data_imputed = imputer.fit_transform(data)
# x = pd.DataFrame(data_imputed, columns = data.columns)
# x.info()

In [5]:
# --- 1. Conversión a cuDF (GPU) y Limpieza Inicial ---
# Asumimos que raw_data es un DataFrame de Pandas.
# Lo convertimos a cuDF al inicio para acelerar el preprocesamiento.
data_gpu = cu_df.DataFrame(raw_data.copy())

# Eliminación de columnas (funciona igual en cuDF)
data_gpu = data_gpu.drop(columns = [
    'CODIGO', 'TIPO', 'FECHA', 'CALIBRE_F', 'MATERIAL_F', 
    'AISLAMIENTO_F', 'G_N', 'CALIBRE_NEUTRO', 'TIPO_TAXONOMIA', 
    'CLASE', 'ALTURA', 'CANTIDAD_TIERRA'
])

# --- 2. Separación de la Variable Objetivo ---
target = ['UITI']
# Extraer 'y' como CuPy array (cp.ndarray)
y = data_gpu[target].to_numpy().astype('float32').ravel() # to_numpy() extrae como cp.array
data_gpu = data_gpu.drop(columns=target)

# --- 3. Imputación de Valores Faltantes (GPU-acelerada) ---
# Usamos SimpleImputer de cuML que opera sobre el cuDF/CuPy.
imputer = SimpleImputer(strategy = 'mean')

# fit_transform opera sobre el cuDF (data_gpu) y devuelve un CuPy array
# El imputador de cuML requiere que los datos de entrada sean numéricos.
data_imputed_cp = imputer.fit_transform(data_gpu) 

# # Volvemos a convertir el array CuPy a un DataFrame de cuDF para usar .info()
# x = cu_df.DataFrame(data_imputed_cp, columns = data_gpu.columns)
# x.info()

## Entrenamiento del modelo

En este caso usaremos un regresor SVM

In [6]:
# # 1. Particionar los datos en conjuntos de entrenamiento y prueba
# # Usaremos 80% para entrenamiento y 20% para prueba.
# X_train, X_test, y_train, y_test = train_test_split(
#     x, y, test_size=0.2, random_state=42
# )

# print(f"Tamaño de X_train: {X_train.shape}")
# print(f"Tamaño de X_test: {X_test.shape}")

# # 2. Escalar los datos (Recomendado para SVR)
# # El SVR es sensible a la escala de las variables, por lo que es crucial estandarizar X.
# # La variable 'y' a menudo NO se escala en regresión, pero puede hacerse.

# scaler_X = StandardScaler()
# X_train_scaled = scaler_X.fit_transform(X_train)
# X_test_scaled = scaler_X.transform(X_test)

# # 3. Entrenar el modelo de Regresión por Vectores de Soporte (SVR)
# # Utilizamos un kernel RBF que es común para SVR.
# svm_regressor = SVR(kernel='rbf', C=100, gamma=0.1, epsilon=.1)
# svm_regressor.fit(X_train_scaled, y_train)

# # 4. Hacer Predicciones
# y_pred = svm_regressor.predict(X_test_scaled)

# # 5. Evaluar el modelo
# # Usaremos el Error Cuadrático Medio (MSE) para evaluar la regresión.
# mse = mean_squared_error(y_test, y_pred)
# print(f"\n Error Cuadrático Medio (MSE) en el conjunto de prueba: {mse:.4f}")

In [7]:
x = data_imputed_cp

# --- 1. Particionar los datos (GPU-acelerado) ---
# Usamos el train_test_split de cuML.
X_train_gpu, X_test_gpu, y_train_gpu, y_test_gpu = train_test_split(
    x, y, test_size=0.2, random_state=42
)

print(f"Partición finalizada. X_train tipo: {type(X_train_gpu)}")
print(f"Tamaño de X_train (GPU): {X_train_gpu.shape}")

# --- 2. Entrenar el modelo de Random Forest Regressor (GPU-acelerado) ---
# Se utiliza el modelo de cuML. Los hiperparámetros por defecto suelen ser un buen inicio.
# n_estimators (número de árboles) es el parámetro principal.

rf_regressor_gpu = RandomForestRegressor(
    n_estimators=1000,      # Número de árboles en el bosque
    max_depth=16,          # Profundidad máxima de los árboles
    random_state=42
)

# El entrenamiento utiliza los datos de CuPy directamente (sin escalar)
rf_regressor_gpu.fit(X_train_gpu, y_train_gpu)

# --- 3. Hacer Predicciones y Evaluar (GPU-acelerado) ---
y_pred_gpu = rf_regressor_gpu.predict(X_test_gpu)

# Evaluación: mean_squared_error de cuML
mse_gpu = mean_squared_error(y_test_gpu, y_pred_gpu)

# Mover la métrica a la CPU para imprimir
mse = cp.asnumpy(mse_gpu) 
print(f"\n Error Cuadrático Medio (MSE): {mse:.4f}")

Partición finalizada. X_train tipo: <class 'cudf.core.dataframe.DataFrame'>
Tamaño de X_train (GPU): (529449, 13)


/usr/local/lib/python3.11/dist-packages/cuml/internals/api_decorators.py:368: UserWarning: For reproducible results in Random Forest Classifier or for almost reproducible results in Random Forest Regressor, n_streams=1 is recommended. If n_streams is > 1, results may vary due to stream/thread timing differences, even when random_state is set
  return init_func(self, *args, **kwargs)



 Error Cuadrático Medio (MSE): 453668.7868
